# Discover DB


## Imports e configurações iniciais

In [2]:
import pandas as pd
import os
import requests
import json
from neo4j import GraphDatabase, basic_auth

## Env Variables

In [3]:
url = 'https://spxa6xmc58.execute-api.us-west-2.amazonaws.com/prod/'
n4j_pass = "nindoo123"
n4j_login = 'neo4j'

## Data loading

In [5]:
def get_crawler(url):
    response = requests.get(url)
    
    return response.text
response = json.loads(get_crawler(url))
#print(response)

## Descobrimento de dados

In [32]:
soma = 0
for element in response:
    #print(element.keys())
    print(element['Category'])
    soma += len(element['Category'])
    # print(element['Description'])
    # print(element['Link'])
    # print(element['PubDate'])
    print(element['Title'])
    # print(element['image'])
print(soma)

['web-development', 'programming', 'graphql', 'api', 'rest-api']
GraphQL in Plain English
['parenting', 'sexuality', 'teenagers', 'christianity', 'relationships']
Our Teens Deserve More than Abstinence-Only Sex Education
['design-patterns', 'ux-design', 'ui', 'design-systems', 'ux']
The Design System Sprint
['computer-science', 'programming', 'data-science', 'software-engineering', 'algorithms']
Introduction to 8 Essential Data Structures
['money', 'culture', 'business', 'education', 'college']
Why I Let My SAT-Tutoring Company Go Under
['white-supremacy', 'identity', 'white-privilege', 'blindspots', 'racial-justice']
Blind Spots, Blond Spots Everywhere
['testing', 'observability', 'steno', 'application-performance', 'satori']
Observability (Re)Done Right!
['virtual-reality', 'gadgets', 'gaming', 'technology', 'features']
Ghost Busting in Spectro VR
['design', 'agile', 'delivery', 'designops']
DesignOps for Agile Delivery
['design', 'software-development', 'product-management', 'ux', '

## Conectando ao db

In [33]:
driver = GraphDatabase.driver( "bolt://localhost:7687",  auth=basic_auth(n4j_login,n4j_pass))
sess = driver.session()

### Interesses:

In [34]:
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            sess.run("""\
                MERGE (a:INTEREST {name: {name}})
                """, {"name":interest})

## Artigos ligados à Interesses

In [39]:
with driver.session() as sess:
    for element in response:
        sess.run("""\
            MERGE (b:Text {title: {title}})
            """, {"title":element['Title']})

In [41]:
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            sess.run("""\
                MATCH (a:INTEREST {name: {name}}),(b:Text {title: {title}})
                MERGE (b)-[r:BELONGS_TO]->(a)
                """, {"name":interest, "title":element['Title']})

In [42]:
with driver.session() as sess:
    for element in response:
        sess.run("""\
                MATCH (b:Text {title: {title}})
                SET b.link = {link}, b.image_url = {image_url}, b.description = {description},                    b.date = {pub_date}
                SET b:Text:Blog
                """, {"link":element['Link'],"image_url":element['image'],"pub_date":element['PubDate'],"description":element['Description'], "title":element['Title']})

### Adicionando features aos artigos

### Conectando áreas similares

In [43]:
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            if len(element['Category']) != 1:    
                sess.run("""\
                    MATCH (a:INTEREST {name: {name}}), (b:INTEREST {name: {other}}) 
                    WHERE NOT (a)-[:IS_RELATED]-(b) 
                    MERGE (a)-[:IS_RELATED]->(b)
                    """, {"name":element['Category'][0], "other":interest})

In [44]:
### Deletando ligações iguais
with driver.session() as sess:
    for element in response:
        for interest in element['Category']:
            if len(element['Category']) >=1:    
                sess.run("""\
                    MATCH (a:INTEREST {name: {name}})-[r:IS_RELATED]->(b:INTEREST {name: {other}}) 
                    WHERE a.name = b.name
                    DELETE r
                    """, {"name":element['Category'][0], "other":interest})

In [8]:
import wikipedia

In [7]:
lista = []
for element in response:
        for interest in element['Category']:
            lista.append(interest)
            interests = set(lista)
print(len(lista))
print(len(interests))
descriptions = {}
for element in interests:
    try:
        descriptions[element] = wikipedia.page(element)
    except:
        descriptions[element] = 'Not Found'

776
381


In [48]:
with open('wiki.json', 'w') as fp:
    json.dump(descriptions, fp)

'Representational state transfer'

In [55]:
ny.content[0:800]

'Representational state transfer (REST) is a software architectural style that defines a set of constraints to be used for creating Web services. Web services that conform to the REST architectural style, called RESTful Web services, provide interoperability between computer systems on the Internet. RESTful Web services allow the requesting systems to access and manipulate textual representations of Web resources by using a uniform and predefined set of stateless operations. Other kinds of Web services, such as SOAP Web services, expose their own arbitrary sets of operations."Web resources" were first defined on the World Wide Web as documents or files identified by their URLs. However, today they have a much more generic and abstract definition that encompasses every thing, entity, or acti'